# OpenADMET PXR Challenge — Final Pipeline
**Kernel:** `oadmet_pxr_tutorial`

Run cells 1-8 in order. Cell 9 (Chemprop) requires switching to `pxr_chemprop311` kernel.

## Cell 1: Imports & Setup

In [ ]:
import os, re, hashlib
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import (AllChem, Descriptors, rdMolDescriptors,
                         MACCSkeys, RDKFingerprint)
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr, kendalltau
from optuna.samplers import TPESampler

optuna.logging.set_verbosity(optuna.logging.WARNING)
BASE  = 'C:/Users/sshec/PXR-Challenge-Tutorial/inputs/'
CACHE = 'feature_cache_final'
os.makedirs(CACHE, exist_ok=True)
print('Imports OK')

## Cell 2: Load & Clean Data

In [ ]:
train_raw      = pd.read_csv(BASE + 'pxr-challenge_TRAIN.csv')
test_unblinded = pd.read_csv(BASE + 'pxr-challenge_TEST_PHASE_1_UNBLINDED.csv')
test_blinded   = pd.read_csv(BASE + 'pxr-challenge_TEST_BLINDED.csv')
counter        = pd.read_csv(BASE + 'pxr-challenge_counter-assay_TRAIN.csv')

labeled_df = train_raw[[
    'Molecule Name','SMILES','OCNT_ID','pEC50',
    'pEC50_std.error (-log10(molarity))',
    'pEC50_ci.lower (-log10(molarity))',
    'pEC50_ci.upper (-log10(molarity))',
    'Emax_estimate (log2FC vs. baseline)',
    'Emax.vs.pos.ctrl_estimate (dimensionless)','Split',
]].copy()
labeled_df = labeled_df.rename(columns={
    'pEC50_std.error (-log10(molarity))':        'pEC50_std_error',
    'pEC50_ci.lower (-log10(molarity))':         'pEC50_ci_lower',
    'pEC50_ci.upper (-log10(molarity))':         'pEC50_ci_upper',
    'Emax_estimate (log2FC vs. baseline)':       'Emax',
    'Emax.vs.pos.ctrl_estimate (dimensionless)': 'Emax_vs_ctrl',
})
for col in ['pEC50','pEC50_std_error','pEC50_ci_lower','pEC50_ci_upper','Emax','Emax_vs_ctrl']:
    labeled_df[col] = pd.to_numeric(labeled_df[col], errors='coerce')
labeled_df = labeled_df.dropna(subset=['SMILES','pEC50']).reset_index(drop=True)

# Counter-assay filter
counter_agg = counter.groupby('OCNT_ID').agg(ca_pEC50_median=('pEC50','median')).reset_index()
labeled_df  = labeled_df.merge(counter_agg, on='OCNT_ID', how='left')
has_counter  = labeled_df['ca_pEC50_median'].notna()
potent       = has_counter & (labeled_df['pEC50'] >= 6)
not_selective= potent & (labeled_df['pEC50'] - labeled_df['ca_pEC50_median'] < 1.5)
labeled_df   = labeled_df[~not_selective].reset_index(drop=True)
labeled_df   = labeled_df[labeled_df['Emax_vs_ctrl'] <= 5].reset_index(drop=True)

has_label = test_unblinded['pEC50'].notna()
y_test    = test_unblinded.loc[has_label,'pEC50'].values.astype(float)
y_train   = labeled_df['pEC50'].values

print(f'Training compounds: {len(labeled_df):,}')
print(f'Unblinded test:     {has_label.sum():,}')
print(f'Blinded test:       {len(test_blinded):,}')

## Cell 3: Feature Functions (Cached)

In [ ]:
def cache_path(smiles):
    s = re.sub(r'\|.*?\|', '', smiles)
    s = re.sub(r'[^A-Za-z0-9._-]', '_', s)
    h = hashlib.sha1(smiles.encode()).hexdigest()[:12]
    return f'{CACHE}/{s[:60]}_{h}.npz'

def compute_3d_conformer(mol, n_confs=10, seed=42):
    params = AllChem.ETKDGv3(); params.randomSeed = seed
    cids = AllChem.EmbedMultipleConfs(mol, numConfs=n_confs, params=params)
    if not cids: return None
    try: AllChem.MMFFOptimizeMoleculeConfs(mol)
    except: pass
    return mol

def compute_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    feats = {}
    fp2 = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
    fp3 = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius=3, nBits=2048)
    for i in range(2048):
        feats[f'morgan_r2_{i}'] = int(fp2.GetBit(i))
        feats[f'morgan_r3_{i}'] = int(fp3.GetBit(i))
    maccs = MACCSkeys.GenMACCSKeys(mol); arr = np.zeros(167,dtype=int)
    Chem.DataStructs.ConvertToNumpyArray(maccs, arr)
    for i in range(167): feats[f'maccs_{i}'] = int(arr[i])
    rdkfp = RDKFingerprint(mol, maxPath=6, fpSize=2048); arr2 = np.zeros(2048,dtype=int)
    Chem.DataStructs.ConvertToNumpyArray(rdkfp, arr2)
    for i in range(2048): feats[f'rdkitfp_{i}'] = int(arr2[i])
    feats.update({
        'rdkit_MolWt': Descriptors.MolWt(mol), 'rdkit_MolLogP': Descriptors.MolLogP(mol),
        'rdkit_TPSA': Descriptors.TPSA(mol), 'rdkit_NumHDonors': Descriptors.NumHDonors(mol),
        'rdkit_NumHAcceptors': Descriptors.NumHAcceptors(mol),
        'rdkit_NumRotatableBonds': Descriptors.NumRotatableBonds(mol),
        'rdkit_RingCount': Descriptors.RingCount(mol),
        'rdkit_FractionCSP3': Descriptors.FractionCSP3(mol),
    })
    mol3d = compute_3d_conformer(Chem.AddHs(mol))
    _3d = ['rdkit_PMI1','rdkit_PMI2','rdkit_PMI3','rdkit_NPR1','rdkit_NPR2',
           'rdkit_RadiusOfGyration','rdkit_Asphericity','rdkit_Eccentricity',
           'rdkit_InertialShapeFactor','rdkit_SpherocityIndex']
    if mol3d:
        for name, val in zip(_3d, [
            rdMolDescriptors.CalcPMI1(mol3d,confId=0), rdMolDescriptors.CalcPMI2(mol3d,confId=0),
            rdMolDescriptors.CalcPMI3(mol3d,confId=0), rdMolDescriptors.CalcNPR1(mol3d,confId=0),
            rdMolDescriptors.CalcNPR2(mol3d,confId=0), rdMolDescriptors.CalcRadiusOfGyration(mol3d,confId=0),
            rdMolDescriptors.CalcAsphericity(mol3d,confId=0), rdMolDescriptors.CalcEccentricity(mol3d,confId=0),
            rdMolDescriptors.CalcInertialShapeFactor(mol3d,confId=0), rdMolDescriptors.CalcSpherocityIndex(mol3d,confId=0),
        ]): feats[name] = val
    else:
        for name in _3d: feats[name] = np.nan
    return feats

def build_features(smiles_series, desc=''):
    feat_list = []
    for i, smi in enumerate(smiles_series):
        if i % 200 == 0: print(f'  [{desc}] {i:,}/{len(smiles_series):,}')
        path = cache_path(smi)
        if os.path.exists(path):
            try: feat_list.append(np.load(path,allow_pickle=True)['feats'].item()); continue
            except: pass
        feats = compute_features(smi)
        feat_list.append(feats if feats else {})
        if feats: np.savez_compressed(path, feats=feats)
    print(f'  [{desc}] Done')
    return pd.DataFrame(feat_list)

ALL_FEATURES = (
    [f'morgan_r2_{i}' for i in range(2048)] +
    [f'morgan_r3_{i}' for i in range(2048)] +
    [f'maccs_{i}'     for i in range(167)]  +
    [f'rdkitfp_{i}'   for i in range(2048)] +
    ['rdkit_MolWt','rdkit_MolLogP','rdkit_TPSA','rdkit_NumHDonors','rdkit_NumHAcceptors',
     'rdkit_NumRotatableBonds','rdkit_RingCount','rdkit_FractionCSP3',
     'rdkit_PMI1','rdkit_PMI2','rdkit_PMI3','rdkit_NPR1','rdkit_NPR2',
     'rdkit_RadiusOfGyration','rdkit_Asphericity','rdkit_Eccentricity',
     'rdkit_InertialShapeFactor','rdkit_SpherocityIndex']
)

def to_X(feat_df):
    for col in ALL_FEATURES:
        if col not in feat_df.columns: feat_df[col] = np.nan
    return feat_df[ALL_FEATURES].values.astype(float)

print(f'Total features: {len(ALL_FEATURES):,}')

## Cell 4: Build Features (uses cache — fast on re-run)

In [ ]:
print('Building features...')
train_feat    = build_features(labeled_df['SMILES'],        'TRAIN')
test_unb_feat = build_features(test_unblinded['SMILES'],    'TEST_UNB')
test_bl_feat  = build_features(test_blinded['SMILES'],      'TEST_BL')

X_train    = to_X(train_feat)
X_test_unb = to_X(test_unb_feat.loc[has_label.values].reset_index(drop=True))
X_test_bl  = to_X(test_bl_feat)

print(f'X_train:    {X_train.shape}')
print(f'X_test_unb: {X_test_unb.shape}')
print(f'X_test_bl:  {X_test_bl.shape}')

## Cell 5: Scaffold Split + Optuna Hyperparameter Tuning

In [ ]:
def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    try: return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except: return None

labeled_df['scaffold'] = labeled_df['SMILES'].apply(get_scaffold)
scaffolds = labeled_df['scaffold'].dropna().unique()
np.random.seed(42); np.random.shuffle(scaffolds)
n_train = int(0.8 * len(scaffolds))
train_scaffolds = set(scaffolds[:n_train])
val_scaffolds   = set(scaffolds[n_train:])
train_mask = labeled_df['scaffold'].isin(train_scaffolds)
val_mask   = labeled_df['scaffold'].isin(val_scaffolds)

X_tr = X_train[train_mask.values]; y_tr = y_train[train_mask.values]
X_vl = X_train[val_mask.values];   y_vl = y_train[val_mask.values]
print(f'Scaffold split — train: {len(y_tr):,}  val: {len(y_vl):,}')

def objective(trial):
    params = {
        'objective': 'regression', 'metric': 'rmse', 'verbosity': -1, 'seed': 42,
        'learning_rate':    trial.suggest_float('learning_rate',    0.005, 0.1,  log=True),
        'num_leaves':       trial.suggest_int('num_leaves',         32,    256),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf',   5,     100),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4,   1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4,   1.0),
        'bagging_freq':     trial.suggest_int('bagging_freq',       1,     10),
        'reg_alpha':        trial.suggest_float('reg_alpha',        1e-8,  10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda',       1e-8,  10.0, log=True),
        'max_depth':        trial.suggest_int('max_depth',          3,     12),
    }
    m = lgb.train(params, lgb.Dataset(X_tr, label=y_tr),
                  valid_sets=[lgb.Dataset(X_vl, label=y_vl)],
                  num_boost_round=2000,
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    return np.sqrt(mean_squared_error(y_vl, m.predict(X_vl)))

study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=100, show_progress_bar=True)
print(f'Best RMSE: {study.best_value:.4f}')
print('Best params:', study.best_params)

## Cell 6: Train Final LightGBM on ALL Data

In [ ]:
best_params = {'objective': 'regression', 'metric': 'rmse', 'verbosity': -1, 'seed': 42,
               **study.best_params}

print('Training final LightGBM on ALL training data...')
final_model = lgb.train(
    best_params,
    lgb.Dataset(X_train, label=y_train),
    num_boost_round=2000,
    callbacks=[lgb.log_evaluation(500)]
)

y_pred_unb = final_model.predict(X_test_unb)
y_pred_bl  = final_model.predict(X_test_bl)

mae  = mean_absolute_error(y_test, y_pred_unb)
r2   = r2_score(y_test, y_pred_unb)
sp   = spearmanr(y_test, y_pred_unb).statistic
kt   = kendalltau(y_test, y_pred_unb).statistic

print(f'\n{"="*50}')
print(f'Final LightGBM — Unblinded Test (n={len(y_test)})')
print(f'{"="*50}')
print(f'MAE:       {mae:.4f}')
print(f'R2:        {r2:.4f}')
print(f'Spearman:  {sp:.4f}')
print(f'Kendall t: {kt:.4f}')

# Save predictions
unb_out = test_unblinded[has_label].copy().reset_index(drop=True)
unb_out['pred_pEC50'] = y_pred_unb
unb_out['residual']   = y_test - y_pred_unb
unb_out[['Molecule Name','SMILES','pEC50','pred_pEC50','residual']].to_csv(
    'unblinded_test_predictions_FINAL.csv', index=False)

bl_out = test_blinded[['SMILES','Molecule Name']].copy()
bl_out['pEC50'] = y_pred_bl
bl_out[['SMILES','Molecule Name','pEC50']].to_csv('pxr_lgbm_final_blinded.csv', index=False)

print('\nSaved: unblinded_test_predictions_FINAL.csv')
print('Saved: pxr_lgbm_final_blinded.csv')

## Cell 7: Validation Plots (LightGBM)

In [ ]:
residuals = y_test - y_pred_unb
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'LightGBM Final — Unblinded Test\nMAE={mae:.3f}  R2={r2:.3f}  Spearman={sp:.3f}', fontsize=12)

ax = axes[0]
ax.scatter(y_test, y_pred_unb, alpha=0.4, s=15, color='#5468C8')
lims = [min(y_test.min(), y_pred_unb.min())-0.2, max(y_test.max(), y_pred_unb.max())+0.2]
ax.plot(lims, lims, 'k--', linewidth=1)
ax.set_xlabel('True pEC50'); ax.set_ylabel('Predicted pEC50')
ax.set_title(f'Predicted vs True (R2={r2:.3f})')
ax.set_xlim(lims); ax.set_ylim(lims); ax.spines[['top','right']].set_visible(False)

ax = axes[1]
ax.scatter(y_pred_unb, residuals, alpha=0.4, s=15, color='#E8593C')
ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Predicted pEC50'); ax.set_ylabel('Residual')
ax.set_title('Residuals vs Predicted'); ax.spines[['top','right']].set_visible(False)

ax = axes[2]
sns.histplot(residuals, bins=40, kde=True, ax=ax, color='#5468C8')
ax.axvline(0, color='black', linestyle='--')
ax.set_title(f'Residual Distribution\nbias={residuals.mean():.3f}  std={residuals.std():.3f}')
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('lgbm_final_validation.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 8: Chemprop CLI Training
> **Switch kernel to `pxr_chemprop311` before running this cell**

In [ ]:
import os, subprocess, shutil, pandas as pd

BASE = 'C:/Users/sshec/PXR-Challenge-Tutorial/inputs/'
test_unblinded = pd.read_csv(BASE + 'pxr-challenge_TEST_PHASE_1_UNBLINDED.csv')
test_blinded   = pd.read_csv(BASE + 'pxr-challenge_TEST_BLINDED.csv')
has_label = test_unblinded['pEC50'].notna()

# Save CSVs for CLI
lgbm_df = pd.read_csv('unblinded_test_predictions_FINAL.csv')
train_cp = lgbm_df[['SMILES','pEC50']].copy()

# Use original train for Chemprop
train_raw = pd.read_csv(BASE + 'pxr-challenge_TRAIN.csv')
counter   = pd.read_csv(BASE + 'pxr-challenge_counter-assay_TRAIN.csv')
labeled   = train_raw[['SMILES','pEC50']].dropna().copy()
labeled.columns = ['smiles','pEC50']
labeled.to_csv('chemprop_train.csv', index=False)

test_unblinded.loc[has_label, ['SMILES','pEC50']].rename(
    columns={'SMILES':'smiles'}).to_csv('chemprop_test_unb.csv', index=False)
test_blinded[['SMILES']].rename(
    columns={'SMILES':'smiles'}).to_csv('chemprop_test_bl.csv', index=False)

# Train
if os.path.exists('chemprop_output'): shutil.rmtree('chemprop_output')
subprocess.run([
    'chemprop','train',
    '--data-path','chemprop_train.csv',
    '--smiles-columns','smiles',
    '--target-columns','pEC50',
    '--task-type','regression',
    '--output-dir','chemprop_output',
    '--epochs','100','--patience','20',
    '--split-type','scaffold_balanced',
    '--num-workers','0',
    '--message-hidden-dim','300','--depth','3',
    '--ffn-hidden-dim','300','--ffn-num-layers','2',
], capture_output=False, text=True)

MODEL = 'chemprop_output/model_0/best.pt'
for test_file, pred_file in [
    ('chemprop_test_unb.csv','chemprop_preds_unb.csv'),
    ('chemprop_test_bl.csv', 'chemprop_preds_bl.csv'),
]:
    subprocess.run(['chemprop','predict',
                    '--test-path', test_file,
                    '--smiles-columns','smiles',
                    '--model-path', MODEL,
                    '--preds-path', pred_file,
                    '--num-workers','0'],
                   capture_output=True, text=True)
print('Chemprop training and prediction complete')

## Cell 9: Ensemble + Final Submission
> **Run in `pxr_chemprop311` kernel after Cell 8 completes**

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import spearmanr
import os

BASE = 'C:/Users/sshec/PXR-Challenge-Tutorial/inputs/'
test_unblinded = pd.read_csv(BASE + 'pxr-challenge_TEST_PHASE_1_UNBLINDED.csv')
test_blinded   = pd.read_csv(BASE + 'pxr-challenge_TEST_BLINDED.csv')
has_label = test_unblinded['pEC50'].notna()
y_test    = test_unblinded.loc[has_label,'pEC50'].values.astype(float)

y_lgbm    = pd.read_csv('unblinded_test_predictions_FINAL.csv')['pred_pEC50'].values.astype(float)
y_lgbm_bl = pd.read_csv('pxr_lgbm_final_blinded.csv')['pEC50'].values.astype(float)
y_cp      = pd.read_csv('chemprop_preds_unb.csv')['pEC50'].values.astype(float)
y_cp_bl   = pd.read_csv('chemprop_preds_bl.csv')['pEC50'].values.astype(float)

best_mae = 999; best_w = 0.5; best_pred = None; best_bl = None
print(f'{"Weights":30} {"MAE":>8} {"R2":>8} {"Spearman":>10}')
print('-'*60)
for w in [0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]:
    y_ens = w*y_lgbm + (1-w)*y_cp
    mae_e = mean_absolute_error(y_test, y_ens)
    r2_e  = r2_score(y_test, y_ens)
    sp_e  = spearmanr(y_test, y_ens).statistic
    marker = ' <- best' if mae_e < best_mae else ''
    if mae_e < best_mae:
        best_mae=mae_e; best_w=w
        best_pred=y_ens.copy()
        best_bl=w*y_lgbm_bl+(1-w)*y_cp_bl
    label = f'LGBM*{w:.1f}+CP*{1-w:.1f}'
    print(f'  {label:<28} {mae_e:>8.4f} {r2_e:>8.4f} {sp_e:>10.4f}{marker}')

print(f'\nBest: LGBM*{best_w:.1f} + CP*{1-best_w:.1f}')
print(f'MAE={best_mae:.4f}  R2={r2_score(y_test,best_pred):.4f}  Spearman={spearmanr(y_test,best_pred).statistic:.4f}')

submission = test_blinded[['SMILES','Molecule Name']].copy()
submission['pEC50'] = best_bl
submission[['SMILES','Molecule Name','pEC50']].to_csv('pxr_submission_best_ensemble.csv', index=False)
print(f'\nSaved: pxr_submission_phase2_final-20260602.csv ({len(submission)} rows)')
print('Submit at: https://huggingface.co/spaces/openadmet/pxr-challenge')